# 9장 — 그림만 보면 틀린다: 인과지도의 규칙과 함정

웹 챕터(9장)와 **같은 예제, 같은 수치**를 파이썬으로 계산합니다:
인과지도를 '부호 붙은 그래프'로 담아 루프를 찾고 극성을 판정하기, 그리고 같은 그림이 숫자에 따라 다르게 움직이는 것.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch9')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 인과지도 = 부호 붙은 화살표 묶음

사내 AI 튜터 도입의 인과지도(웹 실험 ①)를 사전으로 담습니다. 루프 안 − 의 개수가 짝수면 강화(R), 홀수면 균형(B).
부호를 곱해서 판정해도 같아요(+1 × −1 …).

In [ ]:
links = {
    ('사용자 수', '성공 사례 공유'): +1,
    ('성공 사례 공유', '신규 가입'): +1,
    ('신규 가입', '사용자 수'): +1,
    ('사용자 수', '서버 부하'): +1,
    ('서버 부하', '응답 지연'): +1,
    ('응답 지연', '만족도'): -1,
    ('만족도', '신규 가입'): +1,
}

def find_loops(links):
    # 모든 단순 루프를 찾는다(작은 그래프용 깊이 우선 탐색). 같은 루프의 회전은 한 번만.
    nodes = sorted({a for a, _ in links} | {b for _, b in links})
    out = {n: [b for (a, b) in links if a == n] for n in nodes}
    loops = set()
    def dfs(start, node, path):
        for nxt in out[node]:
            if nxt == start:
                k = min(range(len(path)), key=lambda i: path[i])     # 회전 정규화
                loops.add(tuple(path[k:] + path[:k]))
            elif nxt not in path:
                dfs(start, nxt, path + [nxt])
    for n in nodes:
        dfs(n, n, [n])
    return sorted(loops, key=len)

def polarity(loop):
    sign = 1
    for a, b in zip(loop, loop[1:] + loop[:1]):
        sign *= links[(a, b)]
    return 'R(강화)' if sign > 0 else 'B(균형)'

loops = find_loops(links)
for lp in loops:
    negs = sum(1 for a, b in zip(lp, lp[1:] + lp[:1]) if links[(a, b)] < 0)
    print(' → '.join(lp + (lp[0],)), f'   − {negs}개  →  {polarity(lp)}')

assert len(loops) == 2
assert [polarity(lp) for lp in loops] == ['R(강화)', 'B(균형)']
print('✅ 입소문 고리 = R, 속도 저하 고리 = B (− 하나: 응답 지연 → 만족도)')

## 2. 화살표 하나가 결론을 뒤집는다

강화 루프 안의 − 두 개 중 하나를 실수로 + 로 그리면? (연습 문제의 '− 두 개짜리 다섯 화살표 루프')

In [ ]:
loop_signs = [+1, -1, +1, -1, +1]          # − 두 개 → 강화
wrong = [+1, +1, +1, -1, +1]               # − 하나를 + 로 잘못 그림
judge = lambda s: 'R' if math.prod(s) > 0 else 'B'
import math
print('맞게 그린 루프:', judge(loop_signs), '  잘못 그린 루프:', judge(wrong))
assert judge(loop_signs) == 'R' and judge(wrong) == 'B'
assert judge([+1, -1, -1, -1, +1]) == 'B'   # − 세 개 → 균형 (본문 잠깐 확인)
print('✅ 극성 하나가 홀짝을 바꿔 루프의 종류(폭주 ↔ 안정)를 뒤집는다')

## 3. 같은 그림, 다른 행동

1장의 모형(강화 + 균형 + 지연)에서 **지연 D만** 바꿉니다. 성장률 0.35, 한계 1,000명, 처음 50명, 48개월, 오일러법 DT = 1/16.
인과지도는 세 경우 모두 똑같아요.

In [ ]:
def run(D, r=0.35, K=1000, S0=50, T=48, dt=1 / 16):
    S = P = S0
    out = [S]
    for _ in range(round(T / dt)):
        inflow = r * S * (1 - P / K)
        S, P = S + inflow * dt, P + (S - P) / D * dt
        out.append(S)
    return out

peaks = {}
for D in [0.5, 3, 8]:
    path = run(D)
    peaks[D] = max(path)
    i = path.index(peaks[D])
    print(f'지연 {D:3.1f}개월: 최고 {peaks[D]:7.1f}명 ({i / 16:4.1f}개월), 48개월 뒤 {path[-1]:7.1f}명')

assert round(peaks[0.5]) == 1000 and round(peaks[3]) == 1228 and round(peaks[8]) == 2030
print('✅ 같은 인과지도인데 최고 1,000 · 1,228 · 2,030명. 그림만으로는 행동을 알 수 없다')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ts = [i / 16 for i in range(48 * 16 + 1)]
fig, ax = plt.subplots(figsize=(9, 4.4))
for D, c in [(0.5, '#059669'), (3, '#d97706'), (8, '#e11d48')]:
    ax.plot(ts, run(D), color=c, lw=2.2, label=f'지연 {D}개월')
ax.axhline(1000, ls='--', color='#2563eb', lw=1)
ax.set_xlabel('개월'); ax.set_ylabel('사용자 수(명)')
ax.set_title('같은 인과지도(강화 + 균형 + 지연), 숫자 하나만 다른 세 운명')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 오늘의 발견

- **링크 극성**: + 는 "원인이 늘면 결과가 (다른 조건이 같을 때) 원래보다 커짐", − 는 "원래보다 작아짐".
- **루프 극성**: 루프 안 − 가 짝수 개면 강화(R), 홀수 개면 균형(B). 부호를 곱해 판정해도 같다.
- 극성 하나만 틀려도 루프의 종류가 뒤집힌다 → 화살표마다 검수.
- 유량 → 저량의 + 는 "더 쌓인다"이지 "늘어난다"가 아니다(저량 누락의 함정).
- **같은 그림, 다른 행동**: 지연만 바꿔도 최고 1,000 · 1,228 · 2,030명. 결론은 시뮬레이션으로.

다음 장(10장 되풀이되는 이야기)에서 판박이 구조, 시스템 원형을 모아 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**